In [ ]:
!pip install transformers sentence-transformers faiss-cpu datasets torch PyMuPDF -q

In [ ]:
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

In [ ]:
#loading dataset
dataset = load_dataset("neural-bridge/rag-dataset-12000", split="train[:500]")  # Using 500 samples for fast speed

In [ ]:
documents = [item['context'] for item in dataset]
questions = [item['question'] for item in dataset]
answers   = [item['answer']   for item in dataset]

print(f"Total documents: {len(documents)}")
print(f"\nSample document:\n{documents[0][:400]}...")

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedder.encode(documents, show_progress_bar=True, convert_to_numpy=True)

#faiss indexing
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f"Embedding dimension : {dimension}")
print(f"Documents indexed   : {index.ntotal}")

In [ ]:
def retrieve(query, top_k=3):
    """
    Given a query, retrieve the top_k most relevant documents.
    """
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    retrieved_docs = []
    for i, idx in enumerate(indices[0]):
        retrieved_docs.append({
            'rank'    : i + 1,
            'document': documents[idx],
            'distance': distances[0][i]
        })
    return retrieved_docs

test_query = "What is machine learning?"
results = retrieve(test_query, top_k=3)

print(f"Query: {test_query}\n")
for r in results:
    print(f"Rank {r['rank']} (Distance: {r['distance']:.4f})")
    print(f"{r['document'][:200]}...")
    print()

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

print("Loading QA model...")
from transformers import DistilBertTokenizer, DistilBertForQuestionAnswering

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")
model_qa  = DistilBertForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")
model_qa.eval()

print("QA model loaded successfully!")

In [ ]:
def get_answer(question, context):
    """
    Use DistilBERT directly (without pipeline) to extract answer from context.
    """
    inputs = tokenizer(question, context, return_tensors="pt",
                       truncation=True, max_length=512)

    with torch.no_grad():
        outputs = model_qa(**inputs)

    start_idx = torch.argmax(outputs.start_logits)
    end_idx   = torch.argmax(outputs.end_logits) + 1

    input_ids = inputs["input_ids"][0]
    answer    = tokenizer.decode(input_ids[start_idx:end_idx], skip_special_tokens=True)
    return answer


def rag_answer(query, top_k=3):
    """
    Full RAG Pipeline:
    1. Retrieve relevant documents using FAISS
    2. Combine them as context
    3. Generate answer using DistilBERT
    """
    print(f"\n{'='*60}")
    print(f"Question: {query}")
    print(f"{'='*60}")

    retrieved = retrieve(query, top_k=top_k)
    combined_context = " ".join([r['document'] for r in retrieved])[:2000]
    answer = get_answer(query, combined_context)

    print(f"\n Retrieved {top_k} relevant documents")
    print(f"\n Top Retrieved Context (snippet):")
    print(f"  {retrieved[0]['document'][:300]}...")
    print(f"\n Generated Answer : {answer}")
    print(f"{'='*60}\n")

    return answer

In [ ]:
def evaluate_rag(num_samples=10):
    """
    Compare RAG answers against ground truth answers from the dataset.
    """
    print("Evaluating RAG system on sample questions...\n")
    correct = 0

    for i in range(num_samples):
        query       = questions[i]
        true_answer = answers[i].strip().lower()

        # Retrieve relevant documents
        retrieved   = retrieve(query, top_k=3)
        context     = " ".join([r['document'] for r in retrieved])[:2000]

        # Use get_answer() instead of qa_pipeline()
        pred_answer = get_answer(query, context).strip().lower()

        match = true_answer in pred_answer or pred_answer in true_answer
        if match:
            correct += 1

        print(f"Q{i+1}: {query}")
        print(f"  Ground Truth : {true_answer}")
        print(f"  Predicted    : {pred_answer}")
        print(f"  Match        : {'Y' if match else 'N'}\n")

    accuracy = correct / num_samples * 100
    print(f"Accuracy on {num_samples} samples: {accuracy:.1f}%")

evaluate_rag(num_samples=10)

In [ ]:
# Try your own question!
while True:
    user_query = input("\nEnter your question (or 'quit' to exit): ")
    if user_query.lower() == 'quit':
        print("Exiting...")
        break
    rag_answer(user_query, top_k=3)